In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_6.py ──
"""Shared infrastructure for MLFP02 Exercise 6 — Logistic Regression and
Classification Foundations.

Contains: HDB data loading, binary target construction, design matrix
preparation, sigmoid implementation, the neg-log-likelihood + gradient used
by scipy.optimize, calibration-curve binning, and output-directory setup.

Technique-specific code (odds ratios, threshold sweeps, confusion matrices,
ROC/PR/calibration plots, ANOVA + Tukey HSD) does NOT belong here — it
lives in the per-technique files in solutions/ex_6/ and local/ex_6/.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "mlfp02_ex6_logistic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS: list[str] = ["floor_area_sqm", "storey_mid", "remaining_lease"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale transactions (Singapore)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_recent() -> pl.DataFrame:
    """Load HDB resale transactions filtered to 2020+.

    Returns a polars DataFrame with the raw columns from the dataset
    (resale_price, flat_type, floor_area_sqm, storey_range, lease_commence_date,
    month, etc.). No target construction — see build_classification_frame().
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def build_classification_frame(hdb_recent: pl.DataFrame) -> tuple[pl.DataFrame, float]:
    """Add the binary target and engineered features used by logistic regression.

    The target ``high_price`` = 1 if resale_price > median, 0 otherwise — this
    gives a balanced ~50/50 split so the baseline accuracy is 50%.

    Returns (dataframe, median_price).
    """
    median_price = hdb_recent["resale_price"].median()
    frame = hdb_recent.with_columns(
        (pl.col("resale_price") > median_price).cast(pl.Int32).alias("high_price"),
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2.0
        ).alias("storey_mid"),
        (
            99
            - (
                pl.col("month").str.to_date("%Y-%m").dt.year()
                - pl.col("lease_commence_date")
            )
        )
        .cast(pl.Float64)
        .alias("remaining_lease"),
    ).drop_nulls(
        subset=["floor_area_sqm", "storey_mid", "high_price", "remaining_lease"]
    )
    return frame, float(median_price)


def build_design_matrix(
    frame: pl.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Standardise features and append an intercept column.

    Returns (X_with_intercept, y, X_mean, X_std, feature_names_with_intercept).
    """
    X_raw = frame.select(FEATURE_COLS).to_numpy().astype(np.float64)
    y = frame["high_price"].to_numpy().astype(np.float64)

    X_mean = X_raw.mean(axis=0)
    X_std = X_raw.std(axis=0)
    X_scaled = (X_raw - X_mean) / X_std

    n_obs = X_scaled.shape[0]
    X = np.column_stack([np.ones(n_obs), X_scaled])
    feature_names = ["intercept", *FEATURE_COLS]
    return X, y, X_mean, X_std, feature_names


# ════════════════════════════════════════════════════════════════════════
# SIGMOID + BERNOULLI LOG-LIKELIHOOD
# ════════════════════════════════════════════════════════════════════════


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid σ(z) = 1 / (1 + exp(-z)).

    For z ≥ 0 use 1/(1+exp(-z)); for z < 0 use exp(z)/(1+exp(z)). Both
    branches avoid overflow at the extremes.
    """
    z = np.asarray(z, dtype=np.float64)
    result = np.zeros_like(z)
    pos = z >= 0
    neg = ~pos
    result[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg])
    result[neg] = exp_z / (1.0 + exp_z)
    return result


def neg_log_likelihood_logistic(
    beta: np.ndarray, X: np.ndarray, y: np.ndarray
) -> float:
    """Negative Bernoulli log-likelihood: -Σ[y log p + (1-y) log(1-p)]."""
    z = X @ beta
    p = sigmoid(z)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(-ll)


def neg_ll_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Gradient of the negative log-likelihood: -X^T (y - p)."""
    z = X @ beta
    p = sigmoid(z)
    return -X.T @ (y - p)


# ════════════════════════════════════════════════════════════════════════
# ORIGINAL-SCALE COEFFICIENT CONVERSION
# ════════════════════════════════════════════════════════════════════════


def unscale_coefficients(
    beta_scaled: np.ndarray, X_mean: np.ndarray, X_std: np.ndarray
) -> np.ndarray:
    """Convert coefficients fit on standardised features to the original scale.

    β_original[j]   = β_scaled[j] / σ_j  for j ≥ 1
    β_original[0]   = β_scaled[0] - Σ β_scaled[j] * μ_j / σ_j
    """
    beta_original = np.zeros_like(beta_scaled)
    beta_original[0] = beta_scaled[0] - float(np.sum(beta_scaled[1:] * X_mean / X_std))
    beta_original[1:] = beta_scaled[1:] / X_std
    return beta_original


# ════════════════════════════════════════════════════════════════════════
# CALIBRATION BINNING
# ════════════════════════════════════════════════════════════════════════


def calibration_bins(
    y: np.ndarray, p: np.ndarray, n_bins: int = 10
) -> tuple[list[float], list[float], list[int]]:
    """Equal-width binning over predicted probabilities.

    Returns (mean_predicted, mean_observed, counts) for non-empty bins only.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    mean_pred: list[float] = []
    mean_obs: list[float] = []
    counts: list[int] = []
    for i in range(n_bins):
        hi = edges[i + 1] + (1e-12 if i == n_bins - 1 else 0.0)
        mask = (p >= edges[i]) & (p < hi)
        if mask.sum() > 0:
            mean_pred.append(float(p[mask].mean()))
            mean_obs.append(float(y[mask].mean()))
            counts.append(int(mask.sum()))
    return mean_pred, mean_obs, counts




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 6.2: Odds Ratios and Threshold Optimisation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Convert logistic regression coefficients to odds ratios: exp(β)
#   - Interpret odds ratios on the original (unscaled) feature scale
#   - Optimise classification threshold using a domain cost matrix
#   - Compare cost-optimal vs F1-optimal vs default thresholds
#   - Apply threshold optimisation to Singapore HDB valuation risk
#
# PREREQUISITES: Exercise 6.1 — logistic regression from scratch
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — from coefficients to odds ratios
#   2. Build — unscale coefficients + compute odds ratios
#   3. Train — threshold sweep with cost matrix
#   4. Visualise — odds ratio forest plot + cost curve
#   5. Apply — HDB valuation risk: asymmetric error costs
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score



## THEORY — From Coefficients to Odds Ratios


In [ ]:
# Logistic regression models log-odds: log(P/(1-P)) = β₀ + β₁x₁ + ...
#
# The ODDS of an event is P / (1 - P). When P = 0.75, odds = 3:1
# ("three times more likely to happen than not").
#
# An ODDS RATIO for feature j is exp(βⱼ). It answers: "how do the
# odds multiply when xⱼ increases by one unit?"
#
#   exp(βⱼ) > 1  → odds increase (feature promotes the outcome)
#   exp(βⱼ) = 1  → no effect
#   exp(βⱼ) < 1  → odds decrease (feature inhibits the outcome)
#
# For STANDARDISED features, βⱼ represents a one-standard-deviation
# change. To get the per-unit interpretation, we convert back to the
# original scale: β_original[j] = β_scaled[j] / σ_j.
#
# The threshold question is separate from the model. The model outputs
# a probability; the THRESHOLD decides the action. Different costs for
# false positives vs false negatives → different optimal thresholds.



## TASK 2 — BUILD: fit model and compute odds ratios


In [ ]:
print("\n" + "=" * 70)
print("  Odds Ratios and Threshold Optimisation")
print("=" * 70)

# Load data and fit logistic regression (same as 6.1)
hdb_recent = load_hdb_recent()
frame, median_price = build_classification_frame(hdb_recent)
X, y, X_mean, X_std, feature_names = build_design_matrix(frame)
n_obs = X.shape[0]

beta0 = np.zeros(X.shape[1])
result = minimize(
    neg_log_likelihood_logistic,
    beta0,
    args=(X, y),
    method="L-BFGS-B",
    jac=neg_ll_gradient,
    options={"maxiter": 1000, "ftol": 1e-12},
)
beta_scratch = result.x
p_scratch = sigmoid(X @ beta_scratch)

# Convert coefficients to original scale
beta_original = unscale_coefficients(beta_scratch, X_mean, X_std)

print(f"\n=== Odds Ratio Interpretation ===")
print(f"\n{'Feature':<20} {'β (original)':>14} {'Odds Ratio':>12} {'Interpretation'}")
print("─" * 80)
for i in range(1, len(feature_names)):
    or_val = np.exp(beta_original[i])
    name = feature_names[i]
    if or_val > 1:
        interp = f"1-unit increase -> {(or_val-1)*100:.1f}% higher odds"
    else:
        interp = f"1-unit increase -> {(1-or_val)*100:.1f}% lower odds"
    print(f"{name:<20} {beta_original[i]:>14.6f} {or_val:>12.4f} {interp}")

# Practical examples with real-world units
print(f"\n--- Practical Examples ---")
for feat, units, factor in [
    ("floor_area_sqm", "10 sqm", 10),
    ("storey_mid", "5 storeys", 5),
    ("remaining_lease", "10 years", 10),
]:
    idx = feature_names.index(feat)
    or_change = np.exp(beta_original[idx] * factor)
    print(
        f"  +{units} of {feat}: odds multiply by {or_change:.3f} "
        f"({(or_change-1)*100:+.1f}% change)"
    )

# INTERPRETATION: An odds ratio of 1.5 for floor_area_sqm means each
# extra sqm multiplies the odds of being "high price" by 1.5. Odds
# ratios are multiplicative, not additive — they compound with each
# unit increase. A 10 sqm increase multiplies odds by 1.5^10 ≈ 57x.



### Checkpoint 1


In [ ]:
assert np.exp(beta_original[1]) > 1, "Larger area should increase odds of high price"
print("\n[ok] Checkpoint 1 passed — odds ratios computed\n")



## TASK 3 — TRAIN: threshold sweep with cost matrix


In [ ]:
print(f"\n=== Threshold Optimisation ===")

# Cost matrix for HDB property valuation
# FP (predict high, actually low): buyer overpays → cost = $30K
# FN (predict low, actually high): seller underprices → cost = $50K
cost_fp = 30_000
cost_fn = 50_000

thresholds = np.linspace(0.1, 0.9, 81)
total_costs = []
accuracies = []
f1_scores_list = []

for t in thresholds:
    y_pred_t = (p_scratch >= t).astype(int)
    cm = confusion_matrix(y, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
    cost = fp * cost_fp + fn * cost_fn
    total_costs.append(cost)
    accuracies.append(accuracy_score(y, y_pred_t))
    f1_scores_list.append(f1_score(y, y_pred_t, zero_division=0))

optimal_idx = np.argmin(total_costs)
optimal_threshold = thresholds[optimal_idx]
optimal_cost = total_costs[optimal_idx]

# F1-optimal threshold
f1_idx = np.argmax(f1_scores_list)
f1_threshold = thresholds[f1_idx]

print(f"Cost matrix: FP=${cost_fp:,}, FN=${cost_fn:,}")
print(f"\nOptimal threshold (min cost): {optimal_threshold:.3f}")
print(f"  Total cost: ${optimal_cost:,.0f}")
print(f"  Accuracy at this threshold: {accuracies[optimal_idx]:.4f}")
print(f"\nF1-optimal threshold: {f1_threshold:.3f}")
print(f"  F1 at this threshold: {f1_scores_list[f1_idx]:.4f}")
print(f"\nDefault threshold (0.5):")
print(f"  Cost: ${total_costs[40]:,.0f}")
print(f"  Accuracy: {accuracies[40]:.4f}")

# INTERPRETATION: When FN costs more than FP, the optimal threshold
# is below 0.5 — we'd rather predict "high price" more aggressively
# to avoid missing expensive flats. The threshold should reflect the
# business cost of each type of error, not just accuracy.



### Checkpoint 2


In [ ]:
assert 0 < optimal_threshold < 1, "Optimal threshold must be valid"
assert optimal_cost <= total_costs[40], "Optimal cost must be <= default cost"
print("\n[ok] Checkpoint 2 passed — threshold optimisation completed\n")



## TASK 4 — VISUALISE: odds ratio forest plot + cost curve


In [ ]:
# Plot 1: Odds ratio forest plot
or_values = [np.exp(beta_original[i]) for i in range(1, len(feature_names))]
or_names = feature_names[1:]

fig1 = go.Figure()
fig1.add_trace(
    go.Bar(
        y=or_names,
        x=or_values,
        orientation="h",
        marker_color=["#2ecc71" if v > 1 else "#e74c3c" for v in or_values],
    )
)
fig1.add_vline(x=1.0, line_dash="dash", line_color="grey", annotation_text="No effect")
fig1.update_layout(
    title="Odds Ratios per Feature (original scale, per unit)",
    xaxis_title="Odds Ratio exp(β)",
    yaxis_title="Feature",
)
fig1.write_html(str(OUTPUT_DIR / "odds_ratios.html"))
print(f"Saved: {OUTPUT_DIR / 'odds_ratios.html'}")

# Plot 2: Threshold vs cost + accuracy + F1
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=thresholds,
        y=[c / 1e6 for c in total_costs],
        name="Total Cost ($M)",
    )
)
fig2.add_trace(
    go.Scatter(
        x=thresholds,
        y=accuracies,
        name="Accuracy",
        yaxis="y2",
        line={"dash": "dash"},
    )
)
fig2.add_trace(
    go.Scatter(
        x=thresholds,
        y=f1_scores_list,
        name="F1 Score",
        yaxis="y2",
        line={"dash": "dot"},
    )
)
fig2.add_vline(
    x=optimal_threshold,
    line_dash="dash",
    annotation_text=f"Cost-optimal t={optimal_threshold:.2f}",
)
fig2.add_vline(
    x=0.5,
    line_dash="dot",
    line_color="red",
    annotation_text="Default t=0.5",
)
fig2.update_layout(
    title="Cost, Accuracy, and F1 vs Classification Threshold",
    xaxis_title="Threshold",
    yaxis_title="Total Cost ($M)",
    yaxis2={"title": "Score", "overlaying": "y", "side": "right", "range": [0, 1]},
)
fig2.write_html(str(OUTPUT_DIR / "threshold_cost.html"))
print(f"Saved: {OUTPUT_DIR / 'threshold_cost.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — odds ratios + cost curve visualised\n")



## TASK 5 — APPLY: HDB valuation risk — a DIFFERENT cost matrix


Return (unnecessary reviews, missed cases, total S$) at a threshold.


In [ ]:
# SCENARIO: A mortgage lender's valuation-audit team screens HDB resale
# transactions it is financing. A transaction flagged "high-price"
# triggers a detailed review by a licensed valuer.
#
# Two types of errors carry different (ILLUSTRATIVE) costs:
#   - FP (flagged but normal): unnecessary review costs S$800/case
#   - FN (missed overpriced deal): average over-lending exposure of
#     S$45,000 per case
#
# This is NOT the cost matrix of Task 3 (S$30K vs S$50K). A threshold
# is only "cost-optimal" for the costs it was optimised under, so the
# sweep must be re-run with the audit team's own costs.

review_cost = 800  # S$ per unnecessary review (illustrative)
missed_cost = 45_000  # S$ exposure per missed case (illustrative)

# Re-optimise on a finer grid: steep cost ratios push the optimum low
apply_thresholds = np.round(np.linspace(0.01, 0.99, 99), 2)
apply_costs = []
for t in apply_thresholds:
    y_pred_t = (p_scratch >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y, y_pred_t).ravel()
    apply_costs.append(fp_t * review_cost + fn_t * missed_cost)
apply_idx = int(np.argmin(apply_costs))
apply_threshold = float(apply_thresholds[apply_idx])

# For perfectly calibrated probabilities the optimum is the Bayes
# threshold c_FP / (c_FP + c_FN); a gap from it signals miscalibration
bayes_threshold = review_cost / (review_cost + missed_cost)


def audit_cost(threshold: float) -> tuple[int, int, float]:
    y_pred_t = (p_scratch >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y, y_pred_t).ravel()
    return int(fp_t), int(fn_t), float(fp_t * review_cost + fn_t * missed_cost)


print(f"\n=== Real-World Application: Valuation-Audit Screening ===")
print(f"  Transactions scored (2020+ dataset): {n_obs:,}")
print(f"  Bayes threshold for these costs: {bayes_threshold:.3f}")
rows = [
    ("Default", 0.5),
    ("Task 3 optimum (other costs)", float(optimal_threshold)),
    ("Re-optimised for audit costs", apply_threshold),
]
print(f"\n  {'Threshold rule':<30} {'t':>5} {'Reviews':>9} {'Missed':>8} {'Total S$':>15}")
audit_totals = {}
for label, t in rows:
    fp_t, fn_t, total_t = audit_cost(t)
    audit_totals[label] = total_t
    print(f"  {label:<30} {t:>5.2f} {fp_t:>9,} {fn_t:>8,} {total_t:>15,.0f}")
saving_vs_default = audit_totals["Default"] - audit_totals["Re-optimised for audit costs"]
print(f"\n  Saving vs the default threshold: S${saving_vs_default:,.0f}")



### Checkpoint 4


In [ ]:
assert audit_totals["Re-optimised for audit costs"] <= audit_totals["Default"], (
    "Re-optimised threshold must not cost more than the default"
)
assert audit_totals["Re-optimised for audit costs"] <= audit_totals[
    "Task 3 optimum (other costs)"
], "A threshold tuned for other costs cannot beat one tuned for these costs"
print("\n[ok] Checkpoint 4 passed — threshold re-optimised for the audit costs\n")

# BUSINESS IMPACT: Missing an overpriced deal (FN) costs ~56x an
# unnecessary review (FP), so the cost-minimising threshold sits well
# below 0.5: the team accepts many more reviews to miss fewer
# overpriced deals. Reusing a threshold tuned for a different cost
# matrix leaves money on the table — compare the rows above.
#
# LIMITATIONS:
#   - Cost estimates are simplified. Real costs include legal fees,
#     dispute resolution time, and reputational damage that are hard
#     to quantify precisely.
#   - The cost matrix assumes stationarity. In a rising market, FN
#     costs increase; in a falling market, FP costs increase. The
#     threshold should be recalibrated quarterly.
#   - "High-price" (above median) is a proxy for "overpriced"; a real
#     audit model would target price relative to a fair-value estimate.



## REFLECTION


What you've mastered in this technique:
  ✓ Converting standardised logistic coefficients to per-unit odds ratios
  ✓ Reading odds ratios multiplicatively over multi-unit changes
  ✓ Sweeping the decision threshold under a cost matrix
  ✓ Re-optimising the threshold whenever the cost matrix changes, and
    comparing it with the Bayes threshold c_FP / (c_FP + c_FN)

Next: In 03_classification_metrics.py you'll measure the classifier
with confusion matrices, precision/recall, ROC and PR curves.
